# Design Diagnosis and the Design Lock

You are about to roll a new store layout out to 25 of your 50 stores. The rollout
starts next month and you will measure it for six months. Before anything
happens, you want answers to three questions:

1. Which difference-in-differences estimator should I use on *this* panel?
2. Can this design detect an effect of the size I care about?
3. How do I show afterwards that I analysed the study the way I planned to?

This tutorial walks through the design-stage workflow that answers them:

- `diagnose_design` runs candidate estimators on placebo treatments assigned to
  your own untreated history, and reports each one's size and power.
- `DesignDiagnosisResults.lock` records the estimator you choose as a
  `DesignLock`, a small JSON file.
- `PreTrendsPower` on the diagnosis says how large a diverging trend the
  pre-trend check of the planned design would catch.
- `HonestDiD.power_by_M` on the diagnosis says how large an effect a
  sensitivity analysis of the planned design would still detect.
- `DesignLock.fit` fits the locked analysis once the study has run and checks it
  against the lock.
- `BusinessReport(..., design_reconciliation=...)` states the result of that
  check next to the estimate.

Everything here is simulated, so the notebook runs offline and the true effect
is known. In a real study, `history` is data you already have.

## 1. The history you have

Two years of monthly sales for 50 stores, none of them treated. The noise is
strongly serially correlated (a store that is above its level this month tends
to stay above it next month), as business panels usually are. That matters:
serial correlation is exactly what makes some standard errors too small.

In [1]:
import numpy as np
import pandas as pd

from diff_diff import (
    BusinessReport,
    CallawaySantAnna,
    DesignLock,
    HonestDiD,
    ImputationDiD,
    PreTrendsPower,
    TwoWayFixedEffects,
    diagnose_design,
)

N_STORES, N_HISTORY, N_FOLLOW_UP = 50, 24, 6
N_TREATED = 25
SEED = 34


def simulate_outcomes(n_periods, rho=0.8, seed=SEED):
    """Monthly sales index per store: store level + common trend + AR(1) noise."""
    rng = np.random.default_rng(seed)
    store_level = rng.normal(0.0, 1.0, N_STORES)
    noise = np.empty((N_STORES, n_periods))
    noise[:, 0] = rng.normal(0.0, 1.0, N_STORES)
    for t in range(1, n_periods):
        noise[:, t] = rho * noise[:, t - 1] + rng.normal(0.0, np.sqrt(1 - rho**2), N_STORES)
    months = np.arange(1, n_periods + 1)
    sales = 10.0 + store_level[:, None] + 0.05 * months[None, :] + noise
    return pd.DataFrame(
        {
            "store": np.repeat(np.arange(N_STORES), n_periods),
            "month": np.tile(months, N_STORES),
            "sales": sales.ravel(),
        }
    )


history = simulate_outcomes(N_HISTORY)
print(history.shape)
history.head()

(1200, 3)


,store,month,sales
0,0,1,9.744801
1,0,2,8.807253
2,0,3,9.263918
3,0,4,9.054679
4,0,5,9.857585


## 2. Declare the design and diagnose it

`diagnose_design` needs the design you plan to run, stated as numbers:

- **How many units will be treated**: `n_treated=25`.
- **When, relative to the data you have**: the real rollout is in the future, so
  it cannot be simulated on the history directly. Instead you pick a placebo
  adoption month *inside* the history that leaves the same post window as the
  real study will have. The study will run six months, so the placebo adoption
  is month 19 and the placebo post window is months 19-24.
- **Which effects to inject**: 0.25 and 0.5 index points. You hope the layout is
  worth about 0.5.

Each replicate assigns placebo treatment to 25 randomly chosen stores, adds the
injected effect to their post-period outcomes, and fits every candidate on that
same placebo draw. No real effect is estimated at any point.

Four candidates are compared: two-way fixed effects with unit-clustered standard
errors (the default), the same regression with classical standard errors,
Callaway-Sant'Anna, and the imputation estimator.

In [2]:
result = diagnose_design(
    history,
    outcome="sales",
    unit="store",
    time="month",
    estimators={
        "twfe": TwoWayFixedEffects(),
        "twfe_classical": TwoWayFixedEffects(vcov_type="classical"),
        "cs": CallawaySantAnna(),
        "imp": ImputationDiD(),
    },
    cohort_periods=[19],          # placebo adoption month: six post months, 19-24
    n_treated=N_TREATED,          # the planned number of treated stores
    effect_sizes=[0.25, 0.5],     # effects to inject, in sales-index points
    n_simulations=200,
    power=0.80,                   # also search for the minimum detectable effect
    seed=SEED,
    progress=False,
)
result.print_summary()

         Design Diagnosis (placebo assignments on the supplied panel)         

Units x periods:                 50 x 24
Untreated status:                declared by caller (no first_treat column given; not verified)
Placebo-treated units:           25
Adoption periods (common):       [19]
Units resampled:                 True
Significance level (alpha):      0.050
Seed:                            34

------------------------------------------------------------------------------
                            No-effect runs (size)                             
------------------------------------------------------------------------------
Estimator                     Size  Scored   Lost  Coverage  Size OK       MDE
twfe                          2.5%     200      0     97.5%      yes    0.8418
twfe_classical               32.5%     200      0     67.5%       NO         -
cs                            7.0%     200      0     93.0%      yes    0.6472
imp                           4.0%     200

## 3. Read the diagnosis

**Size comes first.** The top table is the no-effect run: nothing was injected,
so every rejection is a false positive. A test at the 5% level should reject
about 5% of the time.

- `twfe_classical` rejects in **32.5%** of the no-effect replicates. Classical
  standard errors ignore the serial correlation in each store's sales, so they
  are far too small. Its power column looks the best of the four, and it means
  nothing: a test that cries wolf a third of the time also "detects" real
  effects often. The library marks it `Size OK: NO` and reports no MDE for it.
- The other three pass the size check.

**Then power.** Among the candidates that pass, Callaway-Sant'Anna has the
highest power at both effect sizes (**56.0%** at 0.5) and the smallest minimum
detectable effect (**0.65**).

That second number is the important design-stage finding. You hoped for an
effect of 0.5, and this design has roughly a coin flip's chance of detecting it.
An effect has to be about 0.65 before the design detects it 80% of the time.
You learn this before the rollout, when you can still add stores, extend the
measurement window, or accept the risk with open eyes.

The next cell prints the numbers behind the `Size OK` column.

In [3]:
threshold = 0.05 + 1.96 * np.sqrt(0.05 * 0.95 / 200)
print(f"Size threshold with 200 replicates: {threshold:.1%}")
for label, runs in result.results.items():
    print(
        f"{label:<15} size {runs[0].rejection_rate:5.1%}   "
        f"size_controlled={result.size_controlled[label]}   MDE {result.mde[label]:.2f}"
    )

Size threshold with 200 replicates: 8.0%
twfe            size  2.5%   size_controlled=True   MDE 0.84
twfe_classical  size 32.5%   size_controlled=False   MDE nan
cs              size  7.0%   size_controlled=True   MDE 0.65
imp             size  4.0%   size_controlled=True   MDE 0.78


The size check allows for Monte Carlo noise: with 200 replicates a correctly
sized test can easily show 6% or 7%, so the threshold is the nominal level plus
1.96 Monte Carlo standard errors, **8.0%** here. `cs` passes at 7.0%, which is
closer to the threshold than `twfe` or `imp`. Section 10 comes back to what that
does and does not tell you.

**What the size check does not cover.** Placebo treatment is assigned at random,
so parallel trends holds by construction in every replicate. The check is about
*inference* on your panel's error structure. A design can pass it and still be
biased if the real treatment goes to stores on a different trend. Parallel
trends needs its own evidence; see the Parallel Trends Diagnostics and Honest
DiD Sensitivity tutorials.

## 4. Lock the choice

You decide to use Callaway-Sant'Anna: it passes the size check and it has the
most power among the candidates that do. That is a reasonable rule, and it has a
known side effect. When you pick the best of several candidates from one set of
simulations, the winner's numbers are a little too good, because part of why it
won is luck in those particular draws.

`lock()` deals with the luck in the draws. Because several estimators were
compared, it reruns the chosen one alone, on the same history, with new placebo
draws, and the lock reports that rerun. You pass the history again because the
results object does not keep it.

In [4]:
lock = result.lock("cs", data=history, progress=False)
lock.print_summary()

                                 Design Lock                                  

Chosen:                          cs (CallawaySantAnna)
Compared:                        twfe, twfe_classical, cs, imp
Units x periods:                 50 x 24
Placebo-treated units:           25
Adoption periods (common):       [19]
Significance level (alpha):      0.050
Reported numbers from:           rerun of the chosen estimator (new placebo draws, same panel)
Size OK:                         yes
MDE:                             0.6561

------------------------------------------------------------------------------
    Effect   Rejects        Bias        RMSE  Coverage  Scored   Lost
------------------------------------------------------------------------------
         0      6.5%   -0.001805      0.2386     93.5%     200      0
      0.25     22.0%   -0.001805      0.2386     93.5%     200      0
       0.5     55.5%   -0.001805      0.2386     93.5%     200      0

Placebo treatment is assigned at ran

The rerun's numbers are close to the comparison's: a no-effect rejection rate of
**6.5%**, power of **55.5%** at 0.5 and an MDE of 0.66. These are the numbers
to quote for the design.

The lock is plain JSON. It holds the chosen estimator and every one of its
parameters, the declared design, a fingerprint of the history, the comparison
rows of all four candidates, and the rerun.

In [5]:
lock_json = lock.to_json()      # in practice: write this to a file and commit it
print(len(lock_json), "characters of JSON")
print(lock_json[:260], "...")

13074 characters of JSON
{
  "schema_version": "1.0",
  "diff_diff_version": "4.0.0",
  "label": "cs",
  "estimator_name": "CallawaySantAnna",
  "params": {
    "control_group": "never_treated",
    "anticipation": 0,
    "estimation_method": "dr",
    "alpha": 0.05,
    "cluster": nu ...


**The lock does not record when it was made.** Nothing inside the file can prove
that you wrote it before you saw the outcomes. That evidence has to come from
outside: commit the file to version control, attach it to a pre-analysis plan,
or send it to a colleague before the rollout starts. The timestamp of that act
is what shows the design was fixed in advance.

## 5. Plan the pre-trend check

After the rollout you will look at the event study and ask whether the treated
and comparison stores were on the same path *before* treatment. A check that
finds nothing is only reassuring if it could have found something. For a fitted
event study `PreTrendsPower` answers that (tutorial `07_pretrends_power`). With
`event_study=True`, `diagnose_design` lets you ask it now, before any outcome
exists: it keeps the event study of every placebo fit, and `PreTrendsPower`
reads the spread of those coefficients.

Two choices in the next cell:

- **One estimator, a new seed.** `cs` was picked from a comparison, so its
  numbers in that run owe something to luck. For the pre-trend plan, rerun the
  chosen estimator alone with new draws. The lock still comes from the
  comparison run above.
- **`base_period="universal"`.** A pre-trend check compares every earlier month
  with the month just before treatment, and that is what this option estimates.
  With the default, the pre-period coefficients are month-to-month changes
  instead. The post-period estimates are the same either way.

In [6]:
import textwrap
import warnings

with warnings.catch_warnings(record=True) as caught:   # collect the warnings to print them below
    warnings.simplefilter("always")
    plan = diagnose_design(
        history,
        outcome="sales",
        unit="store",
        time="month",
        estimators={"cs": CallawaySantAnna(base_period="universal")},
        cohort_periods=[19],
        n_treated=N_TREATED,
        n_simulations=500,
        seed=SEED + 2,                # new draws: not the ones that informed the choice
        event_study=True,             # keep each placebo fit's event study
        progress=False,
    )
    by_pre_periods = PreTrendsPower().power_by_pre_periods(plan)

columns = ["n_pre_periods", "mdv", "power_no_violation", "placebo_rejection_rate"]
print(by_pre_periods[columns].head(8).round(3).to_string(index=False))
for warning in caught:
    print()
    print(textwrap.fill("Warning: " + str(warning.message), width=88))

 n_pre_periods   mdv  power_no_violation  placebo_rejection_rate
             1 0.616               0.050                   0.056
             2 0.357               0.084                   0.092
             3   NaN               0.111                   0.152
             4   NaN               0.125                   0.176
             5   NaN               0.139                   0.196
             6   NaN               0.155                   0.208
             7   NaN               0.165                   0.216
             8   NaN               0.177                   0.230

significant at 0.05) rejected 29.6% of the 500 no-effect replicates, against 25.0%
assumed by the pre-trend power formula. PreTrendsPower repeats this comparison for the
pre-periods it plans and reports planned power only where the two agree; a test of fewer
pre-periods may still agree (see PreTrendsPower.power_by_pre_periods).

11, 12, 13, 14, 15, 16, 17] pre-period(s): in the placebo draws those tests rejecte

**Read the table row by row.** Each row is a pre-trend check over the months
closest to the rollout: the last one, the last two, and so on.

- `power_no_violation` is how often the power formula says that check raises a
  false alarm, that is, rejects although nothing is wrong. It grows with the
  number of months because the check fires when any single coefficient is
  significant: **8.4%** for two months.
- `placebo_rejection_rate` is the same rate counted in the 500 placebo draws,
  none of which contains a violation: **9.2%** for two months.
- `mdv` is the smallest divergence the check would catch 80% of the time, in
  sales-index points per month: **0.357** when two months are tested. A smaller
  divergence is caught less than 80% of the time, and less often the smaller it
  is.

**Blank rows are deliberate.** The formula behind `mdv` assumes that each
coefficient's own test rejects 5% of the time when nothing is wrong. The placebo
draws show whether that holds on your panel. For checks over three or more
months it does not: the three-month check rejected **15.2%** of the time in the
draws against **11.1%** assumed. A planned number for such a check would describe
a better-behaved test than the one you would run, so none is given. That is what
the two warnings printed under the table say: one for the check over all 17 pre-period months, one
listing the rows left blank.

The comparison allows for Monte Carlo noise, so a row near the line can come out
differently with another seed, and more replicates make it stricter. With 2,000
replicates every row of this table is blank: with 25 treated stores each
coefficient's own test rejects about 6% of the time, not 5%, and with enough
draws that shows even for a single month. On a panel this size, read the
pre-trend check with that in mind.

**A pre-trend check that does not reject is not evidence that parallel trends
holds.** This plan tells you how large a divergence it could miss. Tutorial
`07_pretrends_power` covers the same question for a fitted event study, and
tutorial `05_honest_did` shows how the conclusion changes if parallel trends is
violated by a given amount.

### Plan the sensitivity analysis

After the rollout you may also want an interval that stays valid if parallel
trends is violated by up to a stated amount `M`. That is what `HonestDiD`
reports (tutorial `05_honest_did`). The protection costs precision: the more
violation you allow, the wider the interval. `HonestDiD.power_by_M` shows the
cost now. It runs the sensitivity analysis on the event study of each of the 500
placebo fits and counts what the interval did. No formula is involved.

The next cell plans two versions of the analysis:

- **Relative magnitudes** (the default). `M` limits how far the gap between
  treated and comparison stores may drift from one month to the next after
  treatment, compared with the largest month-to-month drift before it. `M = 0`
  allows no drift, and `M = 1` allows each month's drift to be as large as the
  largest one before treatment. The limit is per month, so the drift can add
  up over the six months of follow-up.
- **Smoothness** at `M = 0`, which allows the treated and comparison stores to
  be on different straight-line trends.

In [7]:
with warnings.catch_warnings(record=True) as caught:   # collect the warnings to print them below
    warnings.simplefilter("always")
    relative = HonestDiD().power_by_M(
        plan,
        M_grid=[0, 0.5, 1.0],
        effect_size=0.5,              # the effect you care about, in sales-index points
        progress=False,
    )
    smooth = HonestDiD(method="smoothness").power_by_M(plan, M_grid=[0], progress=False)

columns = ["M", "placebo_rejection_rate", "ci_width", "mde", "power"]
print("Relative magnitudes")
print(relative[columns].round(3).to_string(index=False))
print()
print("Smoothness")
print(smooth[columns].round(3).to_string(index=False))
for warning in caught:
    print()
    print(textwrap.fill("Warning: " + str(warning.message), width=88))

Relative magnitudes
  M  placebo_rejection_rate  ci_width   mde  power
0.0                    0.04     0.890 0.652  0.616
0.5                    0.00     2.178 1.311  0.010
1.0                    0.00     3.467 2.046  0.000

Smoothness
  M  placebo_rejection_rate  ci_width  mde  power
0.0                   0.202      0.73  NaN    NaN

check at alpha=0.05 on the placebo draws: M=0 (excludes zero with no effect in 20.2%; 0
of 500 replicates lost). An interval that excludes zero too often when no effect is
present, or that could not be computed on more than 10% of the replicates, has no
meaningful power.


**Read the relative-magnitudes table row by row.** Each row is the sensitivity
analysis at one value of `M`.

- `mde` is the smallest effect the interval would separate from zero in 80% of
  the draws: **0.652** with no drift allowed, **1.311** at `M = 0.5` and
  **2.046** at `M = 1`. Allowing each month's drift to be as large as the
  biggest one before treatment triples the effect you need.
- `power` is how often the interval would separate an effect of 0.5 from zero:
  **61.6%** with no drift allowed, **1.0%** at `M = 0.5`.
- `placebo_rejection_rate` is how often the interval excluded zero in the 500
  placebo draws, where there is no effect: **4.0%** at `M = 0` and never beyond
  it. A valid 95% interval does so at most 5% of the time.
- `ci_width` is the average width of the interval.

**The smoothness row is blank on purpose.** Its interval excluded zero in
**20.2%** of the placebo draws, four times what a 95% interval should. An
interval that finds effects that are not there has no meaningful power, so no
`mde` is given, and the warning under the tables says so. With 25 treated stores
and 17 pre-period months the smoothness interval is not reliable on this panel.
That is worth knowing before you plan to report it.

These are powers for a design in which parallel trends in fact holds. The table
tells you what protection against a violation would cost you in precision. It
does not tell you how large a violation to expect.

## 6. Six months later

The rollout happened. The analysis panel is the 24 months of history followed by
six new months, with the layout live in 25 stores from month 25. We simulate a
true effect of 0.5, the size you hoped for.

Two details matter for what follows:

- The first 24 months of the analysis panel are the same rows the design was
  diagnosed on. The lock holds their fingerprint, and the reconciliation checks
  it.
- The adoption column `first_treat` exists only on the analysis panel. The
  history had no such column, because nothing was treated yet.

In [8]:
TRUE_EFFECT = 0.5   # known here because we simulate; unknown in a real study

analysis = simulate_outcomes(N_HISTORY + N_FOLLOW_UP)
treated_stores = np.random.default_rng(SEED + 1).choice(N_STORES, size=N_TREATED, replace=False)
is_treated = analysis["store"].isin(treated_stores)
analysis["first_treat"] = np.where(is_treated, N_HISTORY + 1, 0)
analysis.loc[is_treated & (analysis["month"] > N_HISTORY), "sales"] += TRUE_EFFECT

# The first 24 months are exactly the history the design was diagnosed on.
same = analysis.loc[analysis["month"] <= N_HISTORY, ["store", "month", "sales"]]
assert same.reset_index(drop=True).equals(history)
print(analysis.shape, "-", int(is_treated.sum() / (N_HISTORY + N_FOLLOW_UP)), "treated stores")

(1500, 4) - 25 treated stores


## 7. Fit the locked analysis

`lock.fit` builds the locked estimator, fits it with the locked outcome, unit
and time columns, and checks the result against the lock on the same data. You
supply the one thing the lock cannot know: which column holds the real adoption
month.

In [9]:
lock = DesignLock.from_json(lock_json)      # read the lock back, as an analyst would
check = lock.fit(analysis, first_treat="first_treat")
check.print_summary()

               Design Reconciliation (analysis vs. design lock)               

Locked estimator:        cs
Status:                  match
Configuration:           fitted from the lock
Checks:                  28 match, 0 deviate, 0 not checked
Lock sha256:             9c08f03e3de0ff1ffa6e0bfbd551875cb24f0f9e1e9e0f5e26760a2d39f18357

'match' means every check that was run agrees with the lock. It covers
the listed items only; it does not say when the design was fixed, and
deviations are not graded.


The status is `match` on all **28 checks**. Each check is one row of
`check.to_dataframe()`: the estimator class, every constructor parameter, the
column names, the fingerprint of the diagnosed months, and the realized design
(50 stores, a balanced panel, 25 treated, one cohort, a six-month post window).

"Fitted from the lock" means the estimator and its configuration are the locked
ones by construction. You did not type them again, so they cannot have drifted.

The estimate itself is in `check.results`, the ordinary results object of the
locked estimator.

In [10]:
estimate = check.results
low, high = estimate.conf_int
print(type(estimate).__name__)
print(f"ATT {estimate.att:.3f}  (SE {estimate.se:.3f}, 95% CI [{low:.3f}, {high:.3f}], p = {estimate.p_value:.4f})")
print(f"True effect in this simulation: {TRUE_EFFECT}")

CallawaySantAnnaResults
ATT 0.393  (SE 0.228, 95% CI [-0.054, 0.839], p = 0.0846)
True effect in this simulation: 0.5


The estimate is **0.393** with a confidence interval that includes zero. The
true effect is 0.5, the interval covers it, and the test does not reject. That
is not a failure of the analysis. It is the outcome the diagnosis told you to
expect about half the time: power at 0.5 was 55.5%.

This is what the design stage is for. A result like this is much easier to
interpret, and to explain to others, when the power calculation was done on your
own data before the study and is on file.

## 8. Report it

Pass the reconciliation to `BusinessReport` (or `DiagnosticReport`) as
`design_reconciliation=`. The report then says, next to the estimate, how the
analysis relates to the lock.

In [11]:
report = BusinessReport(
    estimate,
    design_reconciliation=check,
    outcome_label="the sales index",
    treatment_label="the new store layout",
    auto_diagnostics=False,       # keep this report to the estimate and the reconciliation
)
print(report.summary())

The new store layout increased the sales index by 0.393 (95% CI: -0.0536 to 0.839). Target parameter: overall ATT (cohort-size-weighted average of ATT(g,t)). Statistically, the confidence interval includes zero; the direction is suggestive but not statistically significant. The analysis was fitted from design lock 'cs' and agrees with it on all 28 checks that were run. The lock does not record when it was made. Sample: 1,500 observations (25 treated, 25 control).


In [12]:
full = report.full_report()
start = full.index("## Design Reconciliation")
print(full[start : full.index("\n## ", start + 1)])

## Design Reconciliation

The analysis was fitted from design lock 'cs' and agrees with it on all 28 checks that were run. The lock does not record when it was made.

- status: `match`
- configuration: fitted from the lock
- checks: 28 match, 0 deviate, 0 not checked
- lock sha256: `9c08f03e3de0ff1ffa6e0bfbd551875cb24f0f9e1e9e0f5e26760a2d39f18357`



The report says the analysis was fitted from the lock and agrees with it, and it
says once more that the lock does not record when it was made. It never calls
the analysis "planned in advance": that depends on when you registered the lock
file, which the library cannot see.

The same block is in `report.to_dict()` for tools that read the report as data.

In [13]:
report.to_dict()["design_reconciliation"]

{'status': 'match',
 'declared': False,
 'label': 'cs',
 'lock_sha256': '9c08f03e3de0ff1ffa6e0bfbd551875cb24f0f9e1e9e0f5e26760a2d39f18357',
 'n_checks': 28,
 'n_match': 28,
 'n_deviates': 0,
 'n_not_checked': 0,
 'deviations': [],
 'not_checked': []}

## 9. When the analysis is not the planned one

**A different realized design.** Suppose you had been asked for a readout after
four months instead of six. The estimator is still the locked one, but the study
that ran is not the one that was diagnosed: the power numbers in the lock belong
to a six-month window.

In [14]:
early = lock.fit(analysis[analysis["month"] <= N_HISTORY + 4], first_treat="first_treat")
early.print_summary()

               Design Reconciliation (analysis vs. design lock)               

Locked estimator:        cs
Status:                  deviates
Configuration:           fitted from the lock
Checks:                  27 match, 1 deviate, 0 not checked
Lock sha256:             9c08f03e3de0ff1ffa6e0bfbd551875cb24f0f9e1e9e0f5e26760a2d39f18357

------------------------------------------------------------------------------
Item                          Locked           Realized         Status        
------------------------------------------------------------------------------
n_post_by_cohort              [6]              [4]              deviates      

'match' means every check that was run agrees with the lock. It covers
the listed items only; it does not say when the design was fixed, and
deviations are not graded.


One item deviates: the post window is four months where the lock has six. The
status is `deviates`, and the report would say so. The library does not grade
the deviation. Whether a shorter window matters is for you to argue; the design
stage only makes sure the difference is visible.

**A different configuration.** Now suppose the analysis was fitted by hand, with
a different comparison group from the locked one. For an analysis you fitted
yourself, `lock.reconcile` compares what you tell it (the estimator, the column
names) with the lock.

In [15]:
estimator = CallawaySantAnna(control_group="not_yet_treated")   # not the locked choice
columns = dict(outcome="sales", unit="store", time="month", first_treat="first_treat")
by_hand = estimator.fit(analysis, **columns)
declared = lock.reconcile(estimator, data=analysis, results=by_hand, **columns)
declared.print_summary()

               Design Reconciliation (analysis vs. design lock)               

Locked estimator:        cs
Status:                  deviates
Configuration:           declared to reconcile()
Checks:                  27 match, 1 deviate, 0 not checked
Lock sha256:             9c08f03e3de0ff1ffa6e0bfbd551875cb24f0f9e1e9e0f5e26760a2d39f18357

------------------------------------------------------------------------------
Item                          Locked           Realized         Status        
------------------------------------------------------------------------------
params.control_group          never_treated    not_yet_treated  deviates      

'match' means every check that was run agrees with the lock. It covers
what was declared to reconcile() and the listed items only; it does not
say when the design was fixed, and deviations are not graded.


In [16]:
deviating_report = BusinessReport(
    by_hand,
    design_reconciliation=declared,
    outcome_label="the sales index",
    treatment_label="the new store layout",
    auto_diagnostics=False,
)
print(deviating_report.summary())
print()
for caveat in deviating_report.caveats():
    print(caveat["severity"], "|", caveat["topic"])
print()
print(f"By-hand ATT {by_hand.att:.3f} vs locked ATT {estimate.att:.3f}")

The new store layout increased the sales index by 0.393 (95% CI: -0.0536 to 0.839). Target parameter: overall ATT (cohort-size-weighted average of ATT(g,t)). Statistically, the confidence interval includes zero; the direction is suggestive but not statistically significant. As declared to reconcile(), the analysis deviates from design lock 'cs' on 1 of 28 checks that were run: params.control_group. The lock does not record when it was made. Sample: 1,500 observations (25 treated) with a dynamic not-yet-treated comparison group (the control set varies by cohort and period); 25 never-treated units are also present.

warning | design_deviation

By-hand ATT 0.393 vs locked ATT 0.393


The reconciliation names the parameter that differs, and the report carries a
`design_deviation` caveat. Note the opening words, "As declared to reconcile()":
on this path the library is comparing what you declared, not what it ran itself.

Here the two estimates are the same to three decimals, because with a single
adoption month and never-treated stores present both comparison groups contain
the same stores. The status is still `deviates`. The reconciliation reports
*that* the analysis differs from the lock, never *how much it matters*.

## 10. What this does and does not give you

**The rerun removes one source of optimism, not all of it.** Choosing the best
candidate overstates its power for two reasons: luck in the placebo draws, and
the fact that this candidate happens to suit this particular panel. The rerun
uses new draws, which removes the first. The second remains. In the library's
simulations it was small, up to about 1.4 percentage points of power (see the
Methodology Registry, `DesignDiagnosis`, "Note (lock rerun)").

**A size verdict near the threshold can flip.** `cs` showed 7.0% in the
comparison and 6.5% in the rerun, both under the 8.0% threshold. A candidate
whose true size is slightly above 5% will pass some runs and fail others. In one
of the library's simulation scenarios the rerun failed the size check for 27% of
locks whose comparison had passed. If `lock()` warns that the two verdicts
differ, take it seriously: more replicates give a sharper answer.

**Comparing one candidate at a time hides the comparison.** The rerun happens
only when one `diagnose_design` call compared several estimators. If you run
five separate calls with one estimator each and keep the best, the library
cannot see that, and none of the five locks is corrected.

**The size check is about inference, not identification.** It says nothing
about parallel trends.

**`match` is not a timestamp.** It means the analysis agrees with the lock on
the checks that were run. When the lock was made is shown by where and when you
registered it.

## Summary

| Step | Call | What you get |
|---|---|---|
| Diagnose | `diagnose_design(history, ...)` | Size, power and MDE of each candidate on your own untreated panel |
| Lock | `result.lock(label, data=history)` | A `DesignLock`: the chosen estimator, its configuration and a rerun of its diagnosis |
| Register | `lock.to_json()` | A file to commit or send before the study starts |
| Plan the pre-trend check | `PreTrendsPower().power_by_pre_periods(diagnose_design(..., event_study=True))` | How large a diverging trend the planned pre-trend check would catch, by number of months tested |
| Plan the sensitivity analysis | `HonestDiD().power_by_M(diagnose_design(..., event_study=True))` | The effect a sensitivity analysis would still detect, by the violation `M` it allows |
| Analyse | `lock.fit(analysis, first_treat=...)` | The estimate (`.results`) and its status against the lock (`match` / `deviates`) |
| Report | `BusinessReport(check.results, design_reconciliation=check)` | The estimate with the reconciliation stated next to it |

For an analysis fitted by hand, `lock.to_estimator()` builds the locked
estimator and `lock.reconcile(...)` checks what you declare against the lock.

**Related tutorials:** Power Analysis (power on synthetic data), Parallel Trends
Diagnostics and Honest DiD Sensitivity (the identifying assumption this workflow
does not test).